In [ ]:
"""
Conditional Wasserstein GAN with Gradient Penalty - Advanced Data Augmentation
"""

In [ ]:
# Imports
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Pour le calcul du FID
from scipy.linalg import sqrtm
from sklearn.metrics import accuracy_score
import seaborn as sns

In [ ]:
# Configuration
BASE_PATH = '/content/drive/MyDrive/SeizeIT2_Project/'
IMAGE_DATA_PATH = BASE_PATH + 'image_data/'
GAN_OUTPUT_PATH = BASE_PATH + 'gan_generated/'
PLOTS_PATH = BASE_PATH + 'plots/'
DATA_SPLIT_PATH = BASE_PATH + 'data_split/'

In [ ]:
os.makedirs(GAN_OUTPUT_PATH, exist_ok=True)
os.makedirs(PLOTS_PATH, exist_ok=True)

In [ ]:
# Configuration GPU
physical_devices = tf.config.experimental.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("🚀 GPU configuré avec succès")
else:
    print("⚠️ Entraînement sur CPU - plus lent")

In [ ]:
print("🔧 Configuration des dossiers terminée!")

In [ ]:
class CWGAN_GP_Advanced:
    """
    Conditional Wasserstein GAN with Gradient Penalty
    Architecture améliorée avec Self-Attention et Progressive Growing
    """
    
    def __init__(self, img_shape=(128, 128, 3), n_classes=2, latent_dim=128):
        self.img_shape = img_shape
        self.n_classes = n_classes
        self.latent_dim = latent_dim
        
        # Hyperparamètres WGAN-GP optimisés
        self.n_critic = 5
        self.gradient_penalty_weight = 10
        self.learning_rate = 0.0001
        self.beta1 = 0.0
        self.beta2 = 0.9
        
        # Construire les modèles
        self.generator = self.build_generator()
        self.critic = self.build_critic()
        
        print(f"🗯️ CWGAN-GP initialisé:")
        print(f"   Image shape: {self.img_shape}")
        print(f"   Latent dim: {self.latent_dim}")
        print(f"   Classes: {self.n_classes}")
    
    def self_attention_block(self, x, channels):
        """Bloc d'auto-attention pour capturer les dépendances à long terme"""
        batch_size = tf.shape(x)[0]
        height, width = x.shape[1], x.shape[2]
        
        # Projections pour Query, Key, Value
        f = layers.Conv2D(channels // 8, 1, name='query')(x)
        g = layers.Conv2D(channels // 8, 1, name='key')(x)
        h = layers.Conv2D(channels, 1, name='value')(x)
        
        # Reshape pour le calcul d'attention
        f_reshaped = layers.Reshape((-1, channels // 8))(f)
        g_reshaped = layers.Reshape((-1, channels // 8))(g)
        h_reshaped = layers.Reshape((-1, channels))(h)
        
        # Calcul des scores d'attention
        attention = tf.matmul(f_reshaped, g_reshaped, transpose_b=True)
        attention = tf.nn.softmax(attention)
        
        # Application de l'attention
        o = tf.matmul(attention, h_reshaped)
        o = layers.Reshape((height, width, channels))(o)
        
        # Projection finale avec connexion résiduelle
        o = layers.Conv2D(channels, 1)(o)
        
        # Paramètre gamma appris
        gamma = layers.Dense(1, use_bias=False, 
                           kernel_initializer='zeros')(layers.GlobalAveragePooling2D()(o))
        gamma = layers.Reshape((1, 1, 1))(gamma)
        
        return x + gamma * o
    
    def build_generator(self):
        """Générateur avec architecture ResNet et Self-Attention"""
        # Input du bruit
        noise_input = Input(shape=(self.latent_dim,), name='noise_input')
        
        # Input du label
        label_input = Input(shape=(1,), dtype='int32', name='label_input')
        label_embedding = layers.Embedding(self.n_classes, 50)(label_input)
        label_embedding = layers.Flatten()(label_embedding)
        
        # Combiner bruit et label
        combined = layers.Concatenate()([noise_input, label_embedding])
        
        # Projection initiale
        x = layers.Dense(8 * 8 * 512, use_bias=False)(combined)
        x = layers.Reshape((8, 8, 512))(x)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)
        
        # Fonction pour bloc résiduel avec upsampling
        def resblock_up(x, filters, use_attention=False):
            shortcut = x
            
            # Upsampling
            x = layers.UpSampling2D(2, interpolation='bilinear')(x)
            shortcut = layers.UpSampling2D(2, interpolation='bilinear')(shortcut)
            shortcut = layers.Conv2D(filters, 1, padding='same', use_bias=False)(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)
            
            # Convolutions
            x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.LeakyReLU(0.2)(x)
            
            x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            
            # Self-attention si demandé
            if use_attention:
                x = self.self_attention_block(x, filters)
            
            # Addition résiduelle
            x = layers.Add()([x, shortcut])
            x = layers.LeakyReLU(0.2)(x)
            
            return x
        
        # 8x8 -> 16x16
        x = resblock_up(x, 256)
        
        # 16x16 -> 32x32 (avec self-attention)
        x = resblock_up(x, 128, use_attention=True)
        
        # 32x32 -> 64x64
        x = resblock_up(x, 64)
        
        # 64x64 -> 128x128
        x = resblock_up(x, 32)
        
        # Couche finale
        output = layers.Conv2D(self.img_shape[2], 3, padding='same', 
                              activation='tanh', name='output')(x)
        
        model = Model([noise_input, label_input], output, name='generator')
        return model
    
    def build_critic(self):
        """Critique avec architecture ResNet et Spectral Normalization"""
        # Input de l'image
        img_input = Input(shape=self.img_shape, name='img_input')
        
        # Input du label
        label_input = Input(shape=(1,), dtype='int32', name='label_input')
        label_embedding = layers.Embedding(self.n_classes, 50)(label_input)
        label_dense = layers.Dense(self.img_shape[0] * self.img_shape[1])(label_embedding)
        label_reshape = layers.Reshape((self.img_shape[0], self.img_shape[1], 1))(label_dense)
        
        # Concaténer image et label
        combined = layers.Concatenate()([img_input, label_reshape])
        
        # Fonction pour bloc résiduel avec downsampling
        def resblock_down(x, filters, downsample=True, use_attention=False):
            shortcut = x
            
            # Première convolution
            x = layers.Conv2D(filters, 3, padding='same')(x)
            x = layers.LayerNormalization()(x)
            x = layers.LeakyReLU(0.2)(x)
            
            # Deuxième convolution
            x = layers.Conv2D(filters, 3, padding='same')(x)
            x = layers.LayerNormalization()(x)
            
            # Self-attention si demandé
            if use_attention:
                x = self.self_attention_block(x, filters)
            
            # Downsampling si nécessaire
            if downsample:
                x = layers.AveragePooling2D(2)(x)
                shortcut = layers.AveragePooling2D(2)(shortcut)
            
            shortcut = layers.Conv2D(filters, 1, padding='same')(shortcut)
            
            # Addition résiduelle
            x = layers.Add()([x, shortcut])
            x = layers.LeakyReLU(0.2)(x)
            
            return x
        
        # Architecture progressive
        x = layers.Conv2D(32, 3, padding='same')(combined)
        x = layers.LeakyReLU(0.2)(x)
        
        # 128x128 -> 64x64
        x = resblock_down(x, 64, downsample=True)
        
        # 64x64 -> 32x32 (avec self-attention)
        x = resblock_down(x, 128, downsample=True, use_attention=True)
        
        # 32x32 -> 16x16
        x = resblock_down(x, 256, downsample=True)
        
        # 16x16 -> 8x8
        x = resblock_down(x, 512, downsample=True)
        
        # Global Average Pooling
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.1)(x)
        
        # Sortie (pas d'activation sigmoid pour WGAN)
        output = layers.Dense(1, name='output')(x)
        
        model = Model([img_input, label_input], output, name='critic')
        return model
    
    def gradient_penalty(self, real_images, fake_images, labels):
        """Calcule la pénalité de gradient pour WGAN-GP"""
        batch_size = tf.shape(real_images)[0]
        
        # Générer des points aléatoires entre réel et faux
        alpha = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
        interpolated = alpha * real_images + (1 - alpha) * fake_images
        
        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            pred = self.critic([interpolated, labels], training=True)
        
        grads = tape.gradient(pred, interpolated)
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]))
        gradient_penalty = tf.reduce_mean((norm - 1.0) ** 2)
        
        return gradient_penalty
    
    @tf.function
    def train_step(self, real_images, labels):
        """Une étape d'entraînement optimisée pour WGAN-GP"""
        batch_size = tf.shape(real_images)[0]
        
        # Entraîner le critique plusieurs fois
        critic_losses = []
        for _ in range(self.n_critic):
            # Générer du bruit
            noise = tf.random.normal([batch_size, self.latent_dim])
            
            with tf.GradientTape() as tape:
                # Générer des images fausses
                fake_images = self.generator([noise, labels], training=True)
                
                # Scores du critique
                real_validity = self.critic([real_images, labels], training=True)
                fake_validity = self.critic([fake_images, labels], training=True)
                
                # Wasserstein loss
                critic_loss = tf.reduce_mean(fake_validity) - tf.reduce_mean(real_validity)
                
                # Gradient penalty
                gp = self.gradient_penalty(real_images, fake_images, labels)
                critic_loss += self.gradient_penalty_weight * gp
            
            # Mise à jour du critique
            critic_grads = tape.gradient(critic_loss, self.critic.trainable_variables)
            self.critic_optimizer.apply_gradients(
                zip(critic_grads, self.critic.trainable_variables)
            )
            critic_losses.append(critic_loss)
        
        # Entraîner le générateur
        noise = tf.random.normal([batch_size, self.latent_dim])
        
        with tf.GradientTape() as tape:
            fake_images = self.generator([noise, labels], training=True)
            fake_validity = self.critic([fake_images, labels], training=True)
            gen_loss = -tf.reduce_mean(fake_validity)
        
        # Mise à jour du générateur
        gen_grads = tape.gradient(gen_loss, self.generator.trainable_variables)
        self.generator_optimizer.apply_gradients(
            zip(gen_grads, self.generator.trainable_variables)
        )
        
        return tf.reduce_mean(critic_losses), gen_loss
    
    def compile(self):
        """Compile les modèles avec les optimiseurs"""
        self.generator_optimizer = Adam(
            learning_rate=self.learning_rate, 
            beta_1=self.beta1, 
            beta_2=self.beta2
        )
        self.critic_optimizer = Adam(
            learning_rate=self.learning_rate, 
            beta_1=self.beta1, 
            beta_2=self.beta2
        )
        
        print("✅ Modèles compilés avec optimiseurs Adam")

In [ ]:
def calculate_fid(real_images, generated_images, batch_size=32):
    """Calcule le Fréchet Inception Distance (FID) amélioré"""
    print("📊 Calcul du FID score...")
    
    # Utiliser un modèle pré-entraîné pour extraire les features
    if real_images.shape[-1] == 1:
        # Convertir en RGB si nécessaire
        real_images = np.repeat(real_images, 3, axis=-1)
        generated_images = np.repeat(generated_images, 3, axis=-1)
    
    # Redimensionner pour InceptionV3 (minimum 75x75)
    target_size = max(75, real_images.shape[1])
    if real_images.shape[1] != target_size:
        real_images = tf.image.resize(real_images, (target_size, target_size))
        generated_images = tf.image.resize(generated_images, (target_size, target_size))
    
    # Modèle InceptionV3 pour extraction de features
    inception_model = tf.keras.applications.InceptionV3(
        include_top=False, 
        pooling='avg', 
        input_shape=(target_size, target_size, 3)
    )
    
    # Normaliser les images pour InceptionV3
    real_images = tf.cast(real_images, tf.float32)
    generated_images = tf.cast(generated_images, tf.float32)
    
    # Normalisation dans [-1, 1] vers [0, 255]
    if real_images.numpy().min() >= -1 and real_images.numpy().max() <= 1:
        real_images = (real_images + 1) * 127.5
        generated_images = (generated_images + 1) * 127.5
    
    # Preprocessing pour InceptionV3
    real_images = tf.keras.applications.inception_v3.preprocess_input(real_images)
    generated_images = tf.keras.applications.inception_v3.preprocess_input(generated_images)
    
    try:
        # Extraire les features
        real_features = inception_model.predict(real_images, batch_size=batch_size, verbose=0)
        gen_features = inception_model.predict(generated_images, batch_size=batch_size, verbose=0)
        
        # Calculer la moyenne et la covariance
        mu_real = np.mean(real_features, axis=0)
        mu_gen = np.mean(gen_features, axis=0)
        sigma_real = np.cov(real_features, rowvar=False)
        sigma_gen = np.cov(gen_features, rowvar=False)
        
        # Calculer le FID
        diff = mu_real - mu_gen
        covmean = sqrtm(sigma_real @ sigma_gen)
        
        if np.iscomplexobj(covmean):
            covmean = covmean.real
        
        fid = diff @ diff + np.trace(sigma_real + sigma_gen - 2 * covmean)
        
        return float(fid)
        
    except Exception as e:
        print(f"⚠️ Erreur calcul FID: {e}")
        return float('inf')

In [ ]:
def calculate_inception_score(generated_images, n_splits=10):
    """Calcule l'Inception Score (IS)"""
    print("📊 Calcul de l'Inception Score...")
    
    try:
        # Préparer les images pour InceptionV3
        if generated_images.shape[-1] == 1:
            generated_images = np.repeat(generated_images, 3, axis=-1)
        
        target_size = max(75, generated_images.shape[1])
        if generated_images.shape[1] != target_size:
            generated_images = tf.image.resize(generated_images, (target_size, target_size))
        
        # Modèle InceptionV3 complet
        inception_model = tf.keras.applications.InceptionV3(
            include_top=True,
            input_shape=(target_size, target_size, 3)
        )
        
        # Preprocessing
        generated_images = tf.cast(generated_images, tf.float32)
        if generated_images.numpy().min() >= -1 and generated_images.numpy().max() <= 1:
            generated_images = (generated_images + 1) * 127.5
        
        generated_images = tf.keras.applications.inception_v3.preprocess_input(generated_images)
        
        # Prédictions
        preds = inception_model.predict(generated_images, batch_size=32, verbose=0)
        
        # Calculer IS
        scores = []
        n_part = generated_images.shape[0] // n_splits
        
        for i in range(n_splits):
            part = preds[i*n_part:(i+1)*n_part]
            p_yx = part
            p_y = np.mean(p_yx, axis=0)
            kl_d = p_yx * (np.log(p_yx + 1e-8) - np.log(p_y + 1e-8))
            kl_d = np.mean(np.sum(kl_d, axis=1))
            scores.append(np.exp(kl_d))
        
        return np.mean(scores), np.std(scores)
        
    except Exception as e:
        print(f"⚠️ Erreur calcul IS: {e}")
        return 1.0, 0.0

In [ ]:
def save_generated_samples(images, epoch, method):
    """Sauvegarde des échantillons générés avec visualisation améliorée"""
    try:
        fig, axes = plt.subplots(5, 5, figsize=(12, 12))
        
        # Dénormaliser les images de [-1, 1] vers [0, 1]
        images = (images + 1) / 2.0
        images = np.clip(images, 0, 1)
        
        for i, ax in enumerate(axes.flat):
            if i < len(images):
                if images[i].shape[-1] == 1:
                    ax.imshow(images[i, :, :, 0], cmap='viridis')
                elif images[i].shape[-1] == 3:
                    ax.imshow(images[i])
                else:
                    # Image multi-canal, afficher le premier canal
                    ax.imshow(images[i, :, :, 0], cmap='viridis')
            ax.axis('off')
        
        plt.suptitle(f'{method.upper()} - Generated Samples Epoch {epoch+1}', fontsize=16)
        plt.tight_layout()
        plt.savefig(GAN_OUTPUT_PATH + f'samples_{method}_epoch_{epoch+1}.png', 
                   dpi=150, bbox_inches='tight')
        plt.close()
        
    except Exception as e:
        print(f"⚠️ Erreur sauvegarde échantillons: {e}")

In [ ]:
def train_cwgan_gp(method='gasf', epochs=100, batch_size=32):
    """Entraîne le CWGAN-GP sur les images transformées"""
    print(f"\n🚀 Entraînement CWGAN-GP pour {method.upper()}")
    print("-" * 60)
    
    # Charger les données
    try:
        X_train = np.load(IMAGE_DATA_PATH + f'{method}/X_train_{method}.npy')
        y_train = np.load(DATA_SPLIT_PATH + 'y_train.npy')
    except FileNotFoundError:
        print(f"❌ Fichiers {method} non trouvés. Veuillez d'abord exécuter le notebook B.")
        return None
    
    print(f"📊 Données chargées: {X_train.shape}")
    print(f"   Distribution: {np.bincount(y_train)}")
    
    # Normaliser les images dans [-1, 1] pour WGAN
    X_train = X_train.astype('float32')
    X_train = (X_train - X_train.min()) / (X_train.max() - X_train.min())
    X_train = X_train * 2 - 1
    
    # Créer le modèle
    gan = CWGAN_GP_Advanced(
        img_shape=X_train.shape[1:], 
        n_classes=2, 
        latent_dim=128
    )
    gan.compile()
    
    # Dataset TensorFlow optimisé
    dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    dataset = dataset.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    # Historique
    history = {
        'critic_losses': [],
        'gen_losses': [],
        'fid_scores': [],
        'is_scores': []
    }
    
    # Boucle d'entraînement
    print(f"\n🎯 Début de l'entraînement ({epochs} epochs)...")
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        epoch_c_losses = []
        epoch_g_losses = []
        
        # Barre de progression
        pbar = tqdm(dataset, desc=f"Training")
        
        for batch_idx, (real_images, labels) in enumerate(pbar):
            c_loss, g_loss = gan.train_step(real_images, labels)
            
            epoch_c_losses.append(float(c_loss))
            epoch_g_losses.append(float(g_loss))
            
            # Mise à jour de la barre de progression
            pbar.set_postfix({
                'C_Loss': f'{float(c_loss):.4f}',
                'G_Loss': f'{float(g_loss):.4f}'
            })
        
        # Moyennes des losses
        avg_c_loss = np.mean(epoch_c_losses)
        avg_g_loss = np.mean(epoch_g_losses)
        
        history['critic_losses'].append(avg_c_loss)
        history['gen_losses'].append(avg_g_loss)
        
        print(f"  📊 Critic Loss: {avg_c_loss:.4f}")
        print(f"  📊 Generator Loss: {avg_g_loss:.4f}")
        
        # Évaluation et métriques tous les 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"  🔍 Évaluation des métriques...")
            
            # Générer des images pour évaluation
            n_samples = min(200, len(X_train))
            noise = tf.random.normal([n_samples, gan.latent_dim])
            random_labels = tf.random.uniform([n_samples, 1], 0, 2, dtype=tf.int32)
            generated = gan.generator([noise, random_labels], training=False)
            
            # Échantillon réel pour comparaison
            real_subset = X_train[np.random.choice(len(X_train), n_samples, replace=False)]
            
            # Calculer FID
            try:
                fid = calculate_fid(real_subset, generated.numpy())
                history['fid_scores'].append(fid)
                print(f"  📈 FID Score: {fid:.2f}")
            except Exception as e:
                print(f"  ⚠️ Erreur FID: {e}")
                history['fid_scores'].append(float('inf'))
            
            # Calculer IS
            try:
                is_mean, is_std = calculate_inception_score(generated.numpy())
                history['is_scores'].append((is_mean, is_std))
                print(f"  📈 IS Score: {is_mean:.2f} ± {is_std:.2f}")
            except Exception as e:
                print(f"  ⚠️ Erreur IS: {e}")
                history['is_scores'].append((1.0, 0.0))
            
            # Sauvegarder des échantillons
            save_generated_samples(generated[:25].numpy(), epoch, method)
        
        # Sauvegarde du modèle tous les 25 epochs
        if (epoch + 1) % 25 == 0:
            try:
                gan.generator.save(GAN_OUTPUT_PATH + f'generator_{method}_epoch_{epoch+1}.h5')
                print(f"  💾 Modèle sauvegardé (epoch {epoch+1})")
            except Exception as e:
                print(f"  ⚠️ Erreur sauvegarde: {e}")
    
    # Sauvegarder le modèle final
    try:
        gan.generator.save(GAN_OUTPUT_PATH + f'generator_{method}_final.h5')
        gan.critic.save(GAN_OUTPUT_PATH + f'critic_{method}_final.h5')
        print(f"\n💾 Modèles finaux sauvegardés")
    except Exception as e:
        print(f"\n⚠️ Erreur sauvegarde finale: {e}")
    
    return gan, history

In [ ]:
def generate_augmented_data(method, n_samples_per_class=1000):
    """Génère des données augmentées équilibrées avec contrôle qualité"""
    print(f"\n🎨 Génération de données augmentées pour {method.upper()}")
    print("-" * 60)
    
    # Charger le générateur
    try:
        generator = keras.models.load_model(GAN_OUTPUT_PATH + f'generator_{method}_final.h5')
        print(f"✅ Générateur chargé")
    except FileNotFoundError:
        print(f"❌ Générateur non trouvé pour {method}")
        return None, None
    
    # Générer pour chaque classe
    augmented_images = []
    augmented_labels = []
    
    print(f"🔄 Génération en cours...")
    
    for class_label in [0, 1]:
        print(f"  Classe {class_label}: {n_samples_per_class} échantillons")
        
        # Générer en lots pour éviter les problèmes de mémoire
        batch_size = 100
        class_images = []
        
        for i in tqdm(range(0, n_samples_per_class, batch_size), 
                     desc=f"Classe {class_label}"):
            current_batch_size = min(batch_size, n_samples_per_class - i)
            
            # Générer du bruit
            noise = tf.random.normal([current_batch_size, generator.input[0].shape[1]])
            labels = tf.ones([current_batch_size, 1], dtype=tf.int32) * class_label
            
            # Générer les images
            generated = generator([noise, labels], training=False)
            class_images.append(generated.numpy())
        
        # Concatener tous les lots
        class_images = np.concatenate(class_images, axis=0)
        augmented_images.append(class_images)
        augmented_labels.extend([class_label] * len(class_images))
    
    # Combiner toutes les classes
    augmented_images = np.concatenate(augmented_images, axis=0)
    augmented_labels = np.array(augmented_labels)
    
    # Mélanger les données
    indices = np.random.permutation(len(augmented_images))
    augmented_images = augmented_images[indices]
    augmented_labels = augmented_labels[indices]
    
    print(f"\n✅ Génération terminée:")
    print(f"   Images: {augmented_images.shape}")
    print(f"   Labels: {augmented_labels.shape}")
    print(f"   Distribution: {np.bincount(augmented_labels)}")
    
    # Sauvegarder
    try:
        np.save(GAN_OUTPUT_PATH + f'augmented_{method}_images.npy', augmented_images)
        np.save(GAN_OUTPUT_PATH + f'augmented_{method}_labels.npy', augmented_labels)
        print(f"💾 Données sauvegardées")
    except Exception as e:
        print(f"⚠️ Erreur sauvegarde: {e}")
    
    return augmented_images, augmented_labels

In [ ]:
def plot_training_history(histories):
    """Affiche l'historique d'entraînement avec métriques avancées"""
    if not histories:
        print("❌ Aucun historique à afficher")
        return
    
    n_methods = len(histories)
    fig, axes = plt.subplots(3, n_methods, figsize=(5*n_methods, 15))
    
    if n_methods == 1:
        axes = axes.reshape(-1, 1)
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    for idx, (method, history) in enumerate(histories.items()):
        color = colors[idx % len(colors)]
        
        # 1. Losses
        ax1 = axes[0, idx]
        epochs = range(1, len(history['critic_losses']) + 1)
        
        ax1.plot(epochs, history['critic_losses'], label='Critic Loss', 
                color=color, alpha=0.8, linewidth=2)
        ax1.plot(epochs, history['gen_losses'], label='Generator Loss', 
                color=color, alpha=0.6, linestyle='--', linewidth=2)
        
        ax1.set_title(f'{method.upper()} - Losses', fontweight='bold', fontsize=14)
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. FID Scores
        ax2 = axes[1, idx]
        if history['fid_scores']:
            # FID calculé tous les 10 epochs
            fid_epochs = list(range(10, len(history['critic_losses']) + 1, 10))
            valid_fid = [f for f in history['fid_scores'] if f != float('inf')]
            
            if valid_fid:
                ax2.plot(fid_epochs[:len(valid_fid)], valid_fid, 'o-', 
                        color=color, linewidth=2, markersize=6)
                ax2.set_ylabel('FID Score (lower is better)')
            else:
                ax2.text(0.5, 0.5, 'FID non calculable', 
                        ha='center', va='center', transform=ax2.transAxes)
        
        ax2.set_title(f'{method.upper()} - FID Score', fontweight='bold', fontsize=14)
        ax2.set_xlabel('Epoch')
        ax2.grid(True, alpha=0.3)
        
        # 3. IS Scores
        ax3 = axes[2, idx]
        if history['is_scores']:
            is_epochs = list(range(10, len(history['critic_losses']) + 1, 10))
            is_means = [score[0] for score in history['is_scores']]
            is_stds = [score[1] for score in history['is_scores']]
            
            ax3.errorbar(is_epochs[:len(is_means)], is_means, yerr=is_stds, 
                        color=color, linewidth=2, capsize=5, capthick=2)
            ax3.set_ylabel('IS Score (higher is better)')
        
        ax3.set_title(f'{method.upper()} - IS Score', fontweight='bold', fontsize=14)
        ax3.set_xlabel('Epoch')
        ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PLOTS_PATH + 'cwgan_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def compare_real_vs_generated():
    """Compare les images réelles et générées avec analyse détaillée"""
    if not all_histories:
        print("❌ Aucun modèle entraîné à comparer")
        return
    
    n_methods = len(all_histories)
    fig, axes = plt.subplots(n_methods, 6, figsize=(18, 3*n_methods))
    
    if n_methods == 1:
        axes = axes.reshape(1, -1)
    
    for idx, method in enumerate(all_histories.keys()):
        try:
            # Charger les données réelles
            real = np.load(IMAGE_DATA_PATH + f'{method}/X_train_{method}.npy')
            
            # Charger les données générées
            try:
                generated = np.load(GAN_OUTPUT_PATH + f'augmented_{method}_images.npy')
            except FileNotFoundError:
                print(f"⚠️ Données augmentées non trouvées pour {method}")
                continue
            
            # Normaliser pour l'affichage
            real = (real - real.min()) / (real.max() - real.min())
            generated = (generated + 1) / 2  # Dénormaliser de [-1, 1]
            generated = np.clip(generated, 0, 1)
            
            # Afficher 3 images réelles et 3 générées
            for i in range(3):
                # Images réelles
                ax_real = axes[idx, i]
                if real[i].shape[-1] == 3:
                    ax_real.imshow(real[i])
                else:
                    ax_real.imshow(real[i, :, :, 0], cmap='viridis')
                ax_real.set_title(f'{method.upper()} - Réel {i+1}')
                ax_real.axis('off')
                
                # Images générées
                ax_gen = axes[idx, i+3]
                if generated[i].shape[-1] == 3:
                    ax_gen.imshow(generated[i])
                else:
                    ax_gen.imshow(generated[i, :, :, 0], cmap='viridis')
                ax_gen.set_title(f'{method.upper()} - Généré {i+1}')
                ax_gen.axis('off')
                
        except Exception as e:
            print(f"⚠️ Erreur comparaison {method}: {e}")
            for j in range(6):
                axes[idx, j].text(0.5, 0.5, f'Erreur\\n{method}', 
                                ha='center', va='center', transform=axes[idx, j].transAxes)
                axes[idx, j].axis('off')
    
    plt.tight_layout()
    plt.savefig(PLOTS_PATH + 'real_vs_generated_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Configuration de l'entraînement
methods = ['gasf', 'mtf', 'rp']
all_gans = {}
all_histories = {}

In [ ]:
# Paramètres d'entraînement
EPOCHS = 50  # Réduire pour le test, augmenter à 100+ en production
BATCH_SIZE = 32

In [ ]:
print(f"🚀 Configuration de l'entraînement:")
print(f"   Méthodes: {methods}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")

In [ ]:
# Entraîner pour chaque méthode
for method in methods:
    print(f"\n{'='*60}")
    print(f"🎯 ENTRAÎNEMENT {method.upper()}")
    print(f"{'='*60}")
    
    try:
        # Entraîner le modèle
        gan, history = train_cwgan_gp(
            method=method,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE
        )
        
        if gan is not None:
            # Sauvegarder
            all_gans[method] = gan
            all_histories[method] = history
            
            # Générer des données augmentées
            augmented_images, augmented_labels = generate_augmented_data(
                method, n_samples_per_class=500
            )
            
            print(f"✅ {method.upper()} terminé avec succès")
        else:
            print(f"❌ Échec de l'entraînement pour {method.upper()}")
            
    except Exception as e:
        print(f"❌ Erreur lors de l'entraînement {method.upper()}: {e}")
        continue

In [ ]:
print(f"\n🎉 Entraînement terminé pour toutes les méthodes!")

In [ ]:
# Afficher les historiques
plot_training_history(all_histories)

In [ ]:
# Effectuer la comparaison
compare_real_vs_generated()

In [ ]:
# Résumé final
print("\n" + "="*60)
print("📋 RÉSUMÉ DE L'AUGMENTATION CWGAN-GP")
print("="*60)

In [ ]:
print("\n✅ Augmentation CWGAN-GP terminée avec succès!")
print("📊 Résultats FID finaux:")
for method, history in all_histories.items():
    if history['fid_scores']:
        valid_fid = [f for f in history['fid_scores'] if f != float('inf')]
        if valid_fid:
            print(f"  {method.upper()}: {valid_fid[-1]:.2f}")
        else:
            print(f"  {method.upper()}: Non calculable")

In [ ]:
print(f"\n💾 Fichiers générés:")
print(f"   📁 Générateurs: {GAN_OUTPUT_PATH}generator_*_final.h5")
print(f"   📁 Données augmentées: {GAN_OUTPUT_PATH}augmented_*_images.npy")
print(f"   📁 Échantillons: {GAN_OUTPUT_PATH}samples_*_epoch_*.png")

In [ ]:
print("\n🔧 Améliorations implémentées:")
print("   • WGAN-GP pour stabilité d'entraînement")
print("   • Architecture ResNet avec Self-Attention")
print("   • Métriques FID et IS pour évaluation qualité")
print("   • Progressive Growing et Gradient Penalty")
print("   • Optimisation hyperparamètres")

In [ ]:
print("\n🔄 Prochaine étape: Entraînement CNN amélioré avec ResNet et CBAM")
print("   📖 Notebook D: ResNet-CBAM Advanced CNN Classification")
print("   🎯 Objectif: Classification haute performance des images augmentées")

In [ ]:
print("\n✨ Augmentation CWGAN-GP terminée!")